In [31]:
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from tavily import TavilyClient
import os
import requests

In [32]:
load_dotenv()
token = os.getenv('TAVILY_API_KEY')
token

'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'

In [33]:
tavily_client = TavilyClient(api_key = token)
response = tavily_client.search('news around amazon in stock market')

In [34]:
response['results'][0]


{'url': 'https://www.perplexity.ai/finance/AMZN',
 'title': 'Amazon.com, Inc. Stock Price: Quote, Forecast, Splits & News (AMZN)',
 'content': "The rally was supported by a favorable macro backdrop — June CPI came in well below expectations at 3.5% year-over-year, easing Fed rate hike fears and lifting growth stocks broadly — alongside company-specific catalysts including Amazon's China warehouse expansion to streamline U.S. customs compliance, a new Electrovaya battery technology partnership for its logistics operations, and the Amazon Leo satellite broadband deal in South Africa. Investor focus remains on Amazon's aggressive AI infrastructure buildout — including a $25 billion bond offering to fund data center expansion — alongside news that OpenAI's GPT-5.6 models are now available on Amazon Bedrock, reinforcing AWS's positioning in the AI platform race. Investor focus remains firmly on AWS's AI infrastructure ambitions: Bank of America and TD Cowen highlighted AWS's accelerating cl

In [35]:
news_link_info =[]
for i in response['results']:
    news_link_info.append([i['url'],i['content']])

In [36]:
news_link_info

[['https://www.perplexity.ai/finance/AMZN',
  "The rally was supported by a favorable macro backdrop — June CPI came in well below expectations at 3.5% year-over-year, easing Fed rate hike fears and lifting growth stocks broadly — alongside company-specific catalysts including Amazon's China warehouse expansion to streamline U.S. customs compliance, a new Electrovaya battery technology partnership for its logistics operations, and the Amazon Leo satellite broadband deal in South Africa. Investor focus remains on Amazon's aggressive AI infrastructure buildout — including a $25 billion bond offering to fund data center expansion — alongside news that OpenAI's GPT-5.6 models are now available on Amazon Bedrock, reinforcing AWS's positioning in the AI platform race. Investor focus remains firmly on AWS's AI infrastructure ambitions: Bank of America and TD Cowen highlighted AWS's accelerating cloud growth ahead of Q2 earnings, while Amazon's recently completed $25 billion bond issuance is v

In [56]:
import requests
from bs4 import BeautifulSoup

def extract_content(news_link_info: list):
    result = []

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:140.0) "
            "Gecko/20100101 Firefox/140.0"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
    }

    session = requests.Session()
    session.headers.update(headers)

    for info in news_link_info:
        url = info[0]

        try:
            response = session.get(url, timeout=15)
            response.raise_for_status()

            soup = BeautifulSoup(response.text, "html.parser")

            # Remove unwanted tags
            for tag in soup(["script", "style", "noscript"]):
                tag.decompose()

            # Extract only paragraph text
            paragraphs = soup.find_all("p")
            text = " ".join(
                p.get_text(strip=True)
                for p in paragraphs
            )

            result.append(text)

        except requests.RequestException as e:
            print(f"Failed to fetch {url}: {e}")
            result.append("")

    return result

In [57]:
extract_content(news_link_info)

Failed to fetch https://www.perplexity.ai/finance/AMZN: 403 Client Error: Forbidden for url: https://www.perplexity.ai/finance/AMZN
Failed to fetch https://finance.yahoo.com/quote/AMZN/news: 404 Client Error: Not Found for url: https://finance.yahoo.com/quote/AMZN/news/


['',
 "AMZN AMZN is tradingin the middleof its 52-week rangeand aboveits 200-day simple moving average. The price of AMZN shares hasdecreased $5.07since the market last closed. This is a1.99% drop. The stock has sincerisen $0.16in after-hours trading. Amazon.com, Inc. is a multinational technology company, which engages in the provision of online retail shopping services. It operates through the following segments: North America, International, and Amazon Web Services (AWS). The North America segment offers retail sale of consumer products, including from sellers, advertising, and subscriptions services through North America-focused online and physical stores. The International segment focuses on retail sale of consumer products, including sellers, advertising, and subscription services through internationally focused online stores. The AWS segment is composed of global sales of computers, storage, databases, and other services for start-ups, enterprises, government agencies, and acade

In [38]:
url

'https://finance.yahoo.com/quote/AMZN/news'